# C04: Funciones y Alcance de Variables

## Objetivos de aprendizaje

Al finalizar esta sesion, seras capaz de:

1. Definir funciones con parametros posicionales, nombrados y valores por defecto.
2. Distinguir entre \*args, \*\*kwargs y los delimitadores / y \* para controlar la forma en que se pasan argumentos.
3. Evitar el anti-patron del default mutable usando None como valor por defecto.
4. Comprender el alcance de variables (Local, Enclosing, Global, Builtins) y usar global / nonlocal correctamente.
5. Trabajar con return multiple, lambdas y funciones de orden superior (map, filter, sorted, higher-order).

## Analogia: Las funciones como recetas de cocina

Imagina que quieres preparar **tacos al pastor**. Cada vez que los haces, sigues los mismos pasos:

| Receta (Funcion) | Detalle |
|---|---|
| **Ingredientes** (Parametros) | Carne, chile, piña, tortillas... |
| **Instrucciones** (Cuerpo) | Marinar, cocinar, servir |
| **Resultado** (Return) | Un plato de tacos listo para comer |

Si cambias un ingrediente (parametro), obtienes un resultado ligeramente distinto. Si no pones nada (valor por defecto), usas la receta clasica. Si la receta esta en tu libreta personal (scope local) solo tu la conoces; si la publicas en un libro de cocina (scope global), cualquiera puede usarla.

Las funciones en Python siguen exactamente esta misma logica: encapsulan logica reutilizable, reciben datos de entrada y producen datos de salida.

## 1. Definicion de funciones

Una funcion se define con `def`, seguida del nombre, parentesis con parametros y dos puntos. El bloque indentado es el cuerpo.

In [ ]:
def saludar(nombre: str, turno: str = "manana") -> str:
    """Retorna un saludo personalizado."""
    return f"Hola {nombre}, buen {turno}!"


print(saludar("Maria"))
print(saludar("Carlos", turno="noche"))

### Elementos clave

- **`def`**: palabra reservada que indica definicion de funcion.
- **Nombre**: convencion snake_case (minúsculas con guion bajo).
- **Parametros**: variables entre parentesis, separadas por comas.
- **Tipo de retorno `-> str`**: type hint opcional pero recomendado.
- **Docstring**: string triple entrecomillado que documenta que hace la funcion.
- **`return`**: devuelve un valor. Sin return explicito, la funcion retorna `None`.

In [ ]:
def calcular_area(largo: float, ancho: float) -> float:
    """Calcula el area de un rectangulo.

    Args:
        largo: dimension en metros.
        ancho: dimension en metros.

    Returns:
        El producto de largo por ancho.
    """
    return largo * ancho


print(calcular_area(5, 3))  # 15.0
print(calcular_area(largo=7, ancho=2))  # 14.0

### Funciones sin return

Si no hay sentencia `return`, la funcion retorna `None` implicitamente. Esto es util para efectos secundarios (imprimir, escribir archivos, etc.).

In [ ]:
def registrar_pago(monto: float) -> None:
    """Imprime el registro de un pago."""
    print(f"Pago registrado: ${monto:.2f}")


resultado = registrar_pago(99.99)
print(f"Valor de retorno: {resultado}")  # None

## 2. Parametros

### 2.1 Parametros posicionales

Se asignan por su **posicion** en la llamada. El orden importa.

In [ ]:
def potencia(base: float, exponente: float) -> float:
    return base ** exponente


print(potencia(2, 10))   # 1024  (base=2, exponente=10)
print(potencia(10, 2))   # 100   (base=10, exponente=2) — el orden cambia el resultado

### 2.2 Parametros nombrados (keyword arguments)

Se especifican explicitamente con `nombre=valor`. El orden no importa.

In [ ]:
print(potencia(exponente=3, base=5))  # 125 — mismo resultado que potencia(5, 3)

### 2.3 Valores por defecto (defaults)

Se asignan con `=` en la definicion. Si el argumento no se pasa, se usa el valor por defecto.

In [ ]:
def crear_usuario(nombre: str, activo: bool = True) -> dict:
    return {"nombre": nombre, "activo": activo}


print(crear_usuario("Ana"))              # {'nombre': 'Ana', 'activo': True}
print(crear_usuario("Luis", activo=False))  # {'nombre': 'Luis', 'activo': False}

### 2.4 \*args y \*\*kwargs

- `*args`: captura un numero **variable** de argumentos posicionales como una **tupla**.
- `**kwargs`: captura un numero **variable** de argumentos nombrados como un **dict**.

In [ ]:
def sumar_todos(*args: float) -> float:
    """Suma todos los argumentos posicionales recibidos."""
    print(f"args recibidos: {args}")  # es una tupla
    return sum(args)


print(sumar_todos(1, 2, 3))          # 6
print(sumar_todos(10, 20, 30, 40))   # 100

In [ ]:
def perfil_usuario(nombre: str, **kwargs) -> dict:
    """Crea un perfil con datos extras como keyword arguments."""
    print(f"kwargs recibidos: {kwargs}")  # es un dict
    return {"nombre": nombre, **kwargs}


print(perfil_usuario("Maria", edad=30, ciudad="CDMX"))

### 2.5 Parametros solo-posicionales (/) y solo-nombrados (*)

Desde Python 3.8, se pueden usar `/` y `*` como delimitadores en la firma:

- Todo lo que esta **a la izquierda de `/`** solo puede pasarse por posicion.
- Todo lo que esta **a la derecha de `*`** solo puede pasarse por nombre.

In [ ]:
def crear_pedido(id_producto: int, /, cantidad: int, *, envio_express: bool = False) -> dict:
    """
    id_producto: solo-posicional (/)
    cantidad: normal (posicional o nombrado)
    envio_express: solo-nombrado (*)
    """
    return {
        "producto": id_producto,
        "cantidad": cantidad,
        "express": envio_express,
    }


# Uso correcto
print(crear_pedido(42, 5, envio_express=True))
print(crear_pedido(42, cantidad=3))

# Esto generaria error:
# crear_pedido(id_producto=42, cantidad=5)  # id_producto es solo-posicional
# crear_pedido(42, 5, True)                 # envio_express es solo-nombrado

### Cuándo usar cada tipo

| Tipo | Cuándo usarlo | Ejemplo |
|---|---|---|
| Posicional | Parametros obligatorios con semantica clara por posicion | `sqrt(x)` |
| Nombrado | Cuando hay multiples booleanos o defaults, para legibilidad | `plot(x, y, color="red")` |
| Default | Valor razonable comun que el usuario puede querer cambiar | `join(sep=", ")` |
| \*args | N variable de items del mismo tipo | `sumar(1, 2, 3)` |
| \*\*kwargs | Opciones de configuracion flexibles | `conectar(host=..., port=...)` |
| / (solo-posicional) | APIs donde el orden es semantico y no debe confundirse | `open(file, mode)` |
| * (solo-nombrado) | Forzar claridad en opciones avanzadas | `sort(*, reverse=True)` |

## 3. Anti-patrones en defaults

### El problema del mutable default

Si usas una **lista, dict o set** como valor por defecto, ese objeto **se crea una sola vez** cuando se define la funcion y se **comparte** entre todas las llamadas. Esto genera bugs difficiles de rastrear.

In [ ]:
# MAL — mutable default
def agregar_item(item: str, lista: list[str] = []) -> list[str]:
    lista.append(item)
    return lista


print(agregar_item("manzana"))   # ['manzana']
print(agregar_item("pera"))     # ['manzana', 'pera'] — la lista se acumula!
print(agregar_item("uva"))      # ['manzana', 'pera', 'uva'] — bug silencioso

### La solucion: usar `None` y crear dentro de la funcion

In [ ]:
# BIEN — None como default
def agregar_item(item: str, lista: list[str] | None = None) -> list[str]:
    if lista is None:
        lista = []
    lista.append(item)
    return lista


print(agregar_item("manzana"))   # ['manzana']
print(agregar_item("pera"))     # ['pera'] — lista nueva cada vez
print(agregar_item("uva"))      # ['uva']

### Por que ocurre esto?

En Python, los valores por defecto se evaluan **una sola vez** al momento de definir la funcion. Si el valor es mutable (list, dict, set), todas las llamadas comparten el **mismo objeto**. `None` es inmutable y seguro porque se reemplaza con un objeto nuevo en cada llamada.

In [ ]:
# Verificacion: los defaults se evaluan una sola vez
import time


def mostrar_timestamp(mensaje: str, ts: float = time.time()) -> None:
    """El timestamp se fija al definir la funcion, NO al llamarla."""
    print(f"{mensaje} — ts: {ts}")


mostrar_timestamp("Primera llamada")
time.sleep(1)
mostrar_timestamp("Segunda llamada")  # mismo timestamp que la primera

## 4. Scope (LEGB)

El **alcance** (scope) determina desde donde Python busca el valor de una variable. Sigue el acronimo **LEGB**:

```
┌─────────────────────────────────────────────────────┐
│  B — Builtins (print, len, sum, range...)           │
│  ┌─────────────────────────────────────────────────┐│
│  │  G — Global (nivel del modulo/archivo)          ││
│  │  ┌─────────────────────────────────────────────┐││
│  │  │  E — Enclosing (funcion exterior)           │││
│  │  │  ┌─────────────────────────────────────────┐│││
│  │  │  │  L — Local (dentro de la funcion actual)││││
│  │  │  └─────────────────────────────────────────┘│││
│  │  └─────────────────────────────────────────────┘││
│  └─────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────┘

  Python busca: Local -> Enclosing -> Global -> Builtins
  Si no encuentra -> NameError
```

| Nivel | Descripcion | Ejemplo |
|---|---|---|
| **Local** | Dentro de la funcion actual | variable creada dentro de `def` |
| **Enclosing** | Funcion exterior (closures) | variable de la funcion que contiene a la actual |
| **Global** | Nivel del archivo/modulo | variable definida fuera de cualquier funcion |
| **Builtins** | Funciones y tipos de Python | `print`, `len`, `int`, `ValueError` |

### 4.1 Scope local vs global

In [ ]:
contador = 0  # global


def incrementar() -> None:
    contador = 10  # local — NO modifica la global
    print(f"Dentro de la funcion: contador = {contador}")


incrementar()
print(f"Fuera de la funcion: contador = {contador}")  # sigue siendo 0

### 4.2 La declaracion `global`

Si necesitas **modificar** una variable global desde dentro de una funcion, usa `global`. Usa esta herramienta con moderacion — puede hacer el codigo dificil de razonar.

In [ ]:
contador_global = 0


def incrementar_global() -> None:
    global contador_global
    contador_global += 1


incrementar_global()
incrementar_global()
incrementar_global()
print(f"Contador global: {contador_global}")  # 3

### 4.3 La declaracion `nonlocal`

Se usa en **closures** (funciones anidadas) para modificar una variable de la funcion **enclosing**.

In [ ]:
def crear_contador() -> callable:
    """Crea un contador encapsulado (closure)."""
    cuenta = 0  # variable enclosing

    def incrementar() -> int:
        nonlocal cuenta
        cuenta += 1
        return cuenta

    return incrementar


mi_contador = crear_contador()
print(mi_contador())  # 1
print(mi_contador())  # 2
print(mi_contador())  # 3

# La variable 'cuenta' no es accesible fuera del closure
# print(cuenta)  # NameError

## 5. Return multiple

### 5.1 Retornar tuplas

En Python, `return a, b` retorna una **tupla** implicitamente. Esto permite devolver multiples valores de forma natural.

In [ ]:
def min_max(lista: list[float]) -> tuple[float, float]:
    """Retorna el minimo y maximo de una lista."""
    return min(lista), max(lista)


resultado = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(resultado)        # (1, 9) — tupla
print(type(resultado))  # <class 'tuple'>

### 5.2 Desempaquetado

Puedes asignar cada valor a una variable distinta usando desempaquetado.

In [ ]:
minimo, maximo = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(f"Minimo: {minimo}, Maximo: {maximo}")

### 5.3 NamedTuple para claridad

Cuando una funcion retorna muchos valores, una tupla anonima puede ser confusa. `NamedTuple` le da **nombres** a cada elemento.

In [ ]:
from typing import NamedTuple


class Estadisticas(NamedTuple):
    promedio: float
    desviacion: float
    minimo: float
    maximo: float


def calcular_estadisticas(datos: list[float]) -> Estadisticas:
    import statistics
    return Estadisticas(
        promedio=statistics.mean(datos),
        desviacion=statistics.stdev(datos),
        minimo=min(datos),
        maximo=max(datos),
    )


est = calcular_estadisticas([10, 20, 30, 40, 50])
print(f"Promedio: {est.promedio}")
print(f"Desviacion: {est.desviacion:.2f}")
print(f"Rango: {est.minimo} - {est.maximo}")

## 6. Lambdas

Una **lambda** es una funcion anonima de una sola expresion. Se usa para operaciones simples donde definir una funcion completa seria excesivo.

In [ ]:
# Funcion regular
def duplicar(x: int) -> int:
    return x * 2


# Equivalente como lambda
duplicar_lambda = lambda x: x * 2

print(duplicar(5))           # 10
print(duplicar_lambda(5))    # 10

### 6.1 Lambdas con `sorted`, `map` y `filter`

In [ ]:
# sorted con key lambda
estudiantes = [("Ana", 88), ("Luis", 95), ("Maria", 72), ("Carlos", 91)]

# Ordenar por calificacion (descendente)
por_calificacion = sorted(estudiantes, key=lambda e: e[1], reverse=True)
print("Por calificacion:", por_calificacion)

In [ ]:
# map: aplicar una funcion a cada elemento
numeros = [1, 2, 3, 4, 5]
cuadrados = list(map(lambda x: x**2, numeros))
print("Cuadrados:", cuadrados)  # [1, 4, 9, 16, 25]

In [ ]:
# filter: filtrar elementos que cumplen una condicion
pares = list(filter(lambda x: x % 2 == 0, numeros))
print("Pares:", pares)  # [2, 4]

### 6.2 Alternativa moderna: list comprehension y generator expression

En muchos casos, una list comprehension es mas legible que `map`/`filter` con lambda.

In [ ]:
# Equivalencias con comprehensions
cuadrados_v2 = [x**2 for x in numeros]
pares_v2 = [x for x in numeros if x % 2 == 0]

print("Cuadrados:", cuadrados_v2)
print("Pares:", pares_v2)

### 6.3 `operator.itemgetter` y `operator.attrgetter`

Para casos comunes de ordenamiento, `operator` ofrece alternativas mas rapidas y legibles que lambda.

In [ ]:
from operator import itemgetter, attrgetter


# itemgetter — extrae un indice o clave
por_nombre = sorted(estudiantes, key=itemgetter(0))
print("Por nombre:", por_nombre)

por_calif = sorted(estudiantes, key=itemgetter(1), reverse=True)
print("Por calificacion:", por_calif)

In [ ]:
# attrgetter — extrae un atributo de un objeto
from dataclasses import dataclass


@dataclass
class Producto:
    nombre: str
    precio: float


catalogo = [
    Producto("Laptop", 1200),
    Producto("Mouse", 25),
    Producto("Teclado", 75),
    Producto("Monitor", 350),
]


por_precio = sorted(catalogo, key=attrgetter("precio"))
for p in por_precio:
    print(f"{p.nombre}: ${p.precio}")

### Cuándo usar lambda vs alternatives

| Escenario | Recomendacion |
|---|---|
| Transformacion simple (x*2, x.name) | `lambda` o list comprehension |
| Filtrado simple | List comprehension con `if` |
| Ordenar por un campo/indice | `operator.itemgetter` o `attrgetter` |
| Logica compleja (condicionales, loops) | Funcion `def` con nombre |
| Reutilizar la misma transformacion | Funcion `def` con nombre |

## 7. Funciones de orden superior

En Python, las **funciones son objetos de primera clase**: se pueden asignar a variables, pasar como argumentos y retornar desde otras funciones. Una **funcion de orden superior** es aquella que:

1. Acepta una funcion como argumento, o
2. Retorna una funcion como resultado.

### 7.1 Funciones como argumentos

In [ ]:
def aplicar_operacion(lista: list[float], operacion: callable) -> list[float]:
    """Aplica una funcion a cada elemento de la lista."""
    return [operacion(x) for x in lista]


datos = [1, 4, 9, 16, 25]

import math

raices = aplicar_operacion(datos, math.sqrt)
print("Raices cuadradas:", [f"{r:.2f}" for r in raices])

absolutos = aplicar_operacion([-3, -1, 2, 5], abs)
print("Absolutos:", absolutos)

### 7.2 Funciones que retornan funciones (closures)

In [ ]:
def crear_multiplicador(factor: float) -> callable:
    """Retorna una funcion que multiplica por el factor dado."""
    def multiplicar(x: float) -> float:
        return x * factor
    return multiplicar


duplicar = crear_multiplicador(2)
triplicar = crear_multiplicador(3)

print(f"Duplicar 5: {duplicar(5)}")    # 10
print(f"Triplicar 5: {triplicar(5)}")  # 15

### 7.3 Ejemplo practical: sistema de filtros configurables

In [ ]:
def crear_filtro(rango_min: float, rango_max: float) -> callable:
    """Crea un filtro que acepta valores dentro del rango dado."""
    def filtro(valor: float) -> bool:
        return rango_min <= valor <= rango_max
    return filtro


temperaturas = [18.5, 22.0, 35.2, 19.8, 24.1, 30.5, 17.3]

filtro_comfort = crear_filtro(20, 25)

comfort = [t for t in temperaturas if filtro_comfort(t)]
print(f"Temperaturas de confort (20-25): {comfort}")

### 7.4 `functools.reduce` — reducir una lista a un solo valor

In [ ]:
from functools import reduce


def factorial(n: int) -> int:
    """Calcula n! usando reduce con un lambda."""
    if n < 0:
        raise ValueError("n debe ser no negativo")
    return reduce(lambda acc, x: acc * x, range(1, n + 1), 1)


print(f"5! = {factorial(5)}")    # 120
print(f"10! = {factorial(10)}")  # 3628800

## Ejercicios

### Ejercicio guiado 1: Validador de contrasenas

Crea una funcion `validar_contrasena` que:
- Acepte la contrasena como primer argumento (solo-posicional con `/`).
- Acepte `min_longitud` (default 8) y `requiere_mayuscula` (default True, solo-nombrado con `*`).
- Retorne una tupla `(es_valida: bool, razon: str)`.

In [ ]:
def validar_contrasena(contrasena: str, /, min_longitud: int = 8, *, requiere_mayuscula: bool = True) -> tuple[bool, str]:
    if len(contrasena) < min_longitud:
        return False, f"Muy corta (minimo {min_longitud} caracteres)"
    if requiere_mayuscula and not any(c.isupper() for c in contrasena):
        return False, "Falta al menos una mayuscula"
    return True, "Contrasena valida"


# Pruebas
pruebas = ["abc", "abcdefgh", "Abcdefgh", "Ab123456", " ABCDEFG"]
for pwd in pruebas:
    valida, razon = validar_contrasena(pwd)
    print(f"'{pwd}' -> {razon}")

### Ejercicio guiado 2: Analisis de datos con NamedTuple

Crea una funcion `analizar_notas` que reciba una lista de diccionarios `{"nombre": str, "nota": float}` y retorne un `NamedTuple` con: promedio, aprobados (count), reprobados (count) y el mejor_estudiante (nombre).

In [ ]:
from typing import NamedTuple


class ResultadoAnalisis(NamedTuple):
    promedio: float
    aprobados: int
    reprobados: int
    mejor_estudiante: str


def analizar_notas(estudiantes: list[dict[str, str | float]]) -> ResultadoAnalisis:
    notas = [e["nota"] for e in estudiantes]
    mejor = max(estudiantes, key=lambda e: e["nota"])
    return ResultadoAnalisis(
        promedio=sum(notas) / len(notas),
        aprobados=sum(1 for n in notas if n >= 6.0),
        reprobados=sum(1 for n in notas if n < 6.0),
        mejor_estudiante=mejor["nombre"],
    )


grupo = [
    {"nombre": "Ana", "nota": 9.2},
    {"nombre": "Luis", "nota": 5.5},
    {"nombre": "Maria", "nota": 8.1},
    {"nombre": "Pedro", "nota": 4.8},
    {"nombre": "Sofia", "nota": 7.5},
]


r = analizar_notas(grupo)
print(f"Promedio: {r.promedio:.2f}")
print(f"Aprobados: {r.aprobados}, Reprobados: {r.reprobados}")
print(f"Mejor estudiante: {r.mejor_estudiante}")

### Ejercicio guiado 3: Closures y encapsulamiento

Crea una funcion `crear_cajero` que retorne tres funciones: `depositar`, `retirar` y `saldo`. El saldo debe estar encapsulado (no accesible directamente desde fuera).

In [ ]:
def crear_cajero(saldo_inicial: float = 0) -> tuple:
    saldo = saldo_inicial

    def depositar(monto: float) -> float:
        nonlocal saldo
        saldo += monto
        return saldo

    def retirar(monto: float) -> float:
        nonlocal saldo
        if monto > saldo:
            print("Fondos insuficientes")
            return saldo
        saldo -= monto
        return saldo

    def ver_saldo() -> float:
        return saldo

    return depositar, retirar, ver_saldo


dep, ret, ver = crear_cajero(1000)
print(f"Saldo inicial: {ver()}")
dep(500)
print(f"Despues de depositar 500: {ver()}")
ret(200)
print(f"Despues de retirar 200: {ver()}")

### Ejercicio independiente: Pipeline de transformaciones

Crea una funcion `pipeline` que acepte un numero arbitrario de funciones (\*funcs) y retorne una nueva funcion que aplique todas en secuencia (de izquierda a derecha).

**Ejemplo de uso:**
```python
doble = lambda x: x * 2
suma_diez = lambda x: x + 10
al_cuadrado = lambda x: x ** 2

mi_pipeline = pipeline(doble, suma_diez, al_cuadrado)
print(mi_pipeline(3))  # ((3*2)+10)^2 = 256
```

**Pista:** Usa `functools.reduce` o un loop para componer las funciones.

In [ ]:
from functools import reduce


def pipeline(*funcs: callable) -> callable:
    """Crea una funcion que aplica todas las funciones en secuencia."""
    def ejecutar(valor):
        return reduce(lambda v, f: f(v), funcs, valor)
    return ejecutar


# Prueba
doble = lambda x: x * 2
suma_diez = lambda x: x + 10
al_cuadrado = lambda x: x ** 2

mi_pipeline = pipeline(doble, suma_diez, al_cuadrado)
print(f"pipeline(3) = {mi_pipeline(3)}")  # 256

# Pipeline de strings
mayusculas = lambda s: s.upper()
reemplazar = lambda s: s.replace(" ", "_")
acortar = lambda s: s[:10]

pipeline_texto = pipeline(mayusculas, reemplazar, acortar)
print(pipeline_texto("hola mundo python"))  # HOLA_MUND

## Resumen

| Tema | Concepto clave |
|---|---|
| **def / return** | `def nombre(params) -> tipo: ...` define una funcion; `return` devuelve un valor |
| **Docstrings** | Documenta el proposito, argumentos y retorno de la funcion |
| **Type hints** | Anotaciones opcionales (`-> float`, `: str`) que mejoran la legibilidad |
| **Parametros posicionales** | Se asignan por orden; el importa |
| **Parametros nombrados** | Se asignan por nombre; el orden no importa |
| **Defaults** | Valores por defecto si el argumento no se pasa |
| **\*args** | Tupla de argumentos posicionales variables |
| **\*\*kwargs** | Dict de argumentos nombrados variables |
| **/ (solo-posicional)** | Parametros que solo pueden pasarse por posicion |
| **\* (solo-nombrado)** | Parametros que solo pueden pasarse por nombre |
| **Mutable default** | NUNCA uses list/dict/set como default; usa `None` y crea dentro |
| **LEGB** | Local -> Enclosing -> Global -> Builtins (orden de busqueda) |
| **global** | Modifica una variable global desde dentro de una funcion |
| **nonlocal** | Modifica una variable de la funcion enclosing (closures) |
| **Return multiple** | `return a, b` retorna una tupla; desempaqueta con `a, b = func()` |
| **NamedTuple** | Tupla con nombres para mayor claridad |
| **Lambda** | Funcion anonima de una expresion: `lambda x: x * 2` |
| **map / filter** | Aplican una funcion a cada elemento / filtran elementos |
| **itemgetter** | Extrae un indice/clave (mas rapido que lambda) |
| **attrgetter** | Extrae un atributo de un objeto |
| **Funciones de orden superior** | Aceptan o retornan funciones (map, filter, closures) |
| **Closure** | Funcion anidada que captura variables de su enclosing scope |